<a href="https://colab.research.google.com/github/lohithshankarappa/bisLAB/blob/main/Week2PSO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random
import copy


COURSES = ["CS101", "CS102", "MATH201", "PHY101", "BIO101", "ENG101"]
PROFESSORS = ["Prof. A", "Prof. B", "Prof. A", "Prof. C", "Prof. D", "Prof. B"]
ROOMS = ["Room 101", "Room 102"]
TIMESLOTS = ["Mon 9:00", "Mon 11:00", "Wed 9:00", "Wed 11:00", "Fri 9:00"]

NUM_COURSES = len(COURSES)
NUM_ROOMS = len(ROOMS)
NUM_TIMESLOTS = len(TIMESLOTS)
DIMENSIONS = NUM_COURSES * 2

# --- 2. FITNESS FUNCTION (Objective) ---
def evaluate_timetable(position):
    """
    Decodes continuous positions into integer IDs for rooms and timeslots.
    Calculates penalties based on hard and soft constraints.
    Lower fitness score = Better timetable (0 is perfect).
    """
    penalties = 0

    # Decode position vector into a structured timetable
    timetable = []
    for i in range(NUM_COURSES):
        # Continuous value mapped to discrete index via clipping and rounding
        room_idx = int(np.clip(round(position[i * 2]), 0, NUM_ROOMS - 1))
        slot_idx = int(np.clip(round(position[i * 2 + 1]), 0, NUM_TIMESLOTS - 1))

        timetable.append({
            "course": COURSES[i],
            "prof": PROFESSORS[i],
            "room": room_idx,
            "slot": slot_idx
        })

    # --- Constraint Checking ---
    for i in range(len(timetable)):
        for j in range(i + 1, len(timetable)):
            t1 = timetable[i]
            t2 = timetable[j]

            if t1["room"] == t2["room"] and t1["slot"] == t2["slot"]:
                penalties += 100

            if t1["prof"] == t2["prof"] and t1["slot"] == t2["slot"]:
                penalties += 100


        if timetable[i]["slot"] == 0:
            penalties += 5

    return penalties

class Particle:
    def __init__(self):

        self.position = np.zeros(DIMENSIONS)
        self.velocity = np.random.uniform(-1, 1, DIMENSIONS)

        for i in range(NUM_COURSES):
            self.position[i * 2] = random.uniform(0, NUM_ROOMS - 1)
            self.position[i * 2 + 1] = random.uniform(0, NUM_TIMESLOTS - 1)

        self.fitness = evaluate_timetable(self.position)
        self.best_position = copy.deepcopy(self.position)
        self.best_fitness = self.fitness

# --- 4. PSO ALGORITHM CORE ---
def pso_timetable_optimizer(pop_size=30, max_iter=100, w=0.729, c1=1.494, c2=1.494):
    # Initialize Swarm
    swarm = [Particle() for _ in range(pop_size)]

    # Initialize Global Best
    global_best_position = copy.deepcopy(swarm[0].position)
    global_best_fitness = swarm[0].fitness

    for particle in swarm:
        if particle.fitness < global_best_fitness:
            global_best_fitness = particle.fitness
            global_best_position = copy.deepcopy(particle.position)

    # Main Optimization Loop
    for it in range(max_iter):
        for particle in swarm:
            # Generate random coefficients
            r1 = np.random.rand(DIMENSIONS)
            r2 = np.random.rand(DIMENSIONS)

            # Update Velocity Formula: w*v + c1*r1*(pbest - x) + c2*r2*(gbest - x)
            cognitive = c1 * r1 * (particle.best_position - particle.position)
            social = c2 * r2 * (global_best_position - particle.position)
            particle.velocity = w * particle.velocity + cognitive + social

            # Update Position
            particle.position += particle.velocity

            # Bound handling (keep particles within valid room/timeslot spectrums)
            for i in range(NUM_COURSES):
                particle.position[i * 2] = np.clip(particle.position[i * 2], 0, NUM_ROOMS - 1)
                particle.position[i * 2 + 1] = np.clip(particle.position[i * 2 + 1], 0, NUM_TIMESLOTS - 1)

            # Evaluate Fitness
            particle.fitness = evaluate_timetable(particle.position)

            # Update Personal Best
            if particle.fitness < particle.best_fitness:
                particle.best_fitness = particle.fitness
                particle.best_position = copy.deepcopy(particle.position)

            # Update Global Best
            if particle.fitness < global_best_fitness:
                global_best_fitness = particle.fitness
                global_best_position = copy.deepcopy(particle.position)

        # Early break if ideal conflict-free timetable found (Soft constraints might still apply)
        if global_best_fitness == 0:
            break

        print(f"Iteration {it+1}/{max_iter} -> Best Penalty Score: {global_best_fitness}")

    return global_best_position, global_best_fitness

def print_timetable(best_position):
    print("\n==== OPTIMIZED UNIVERSITY TIMETABLE ====")
    print(f"{'Course':<10} | {'Professor':<12} | {'Room':<10} | {'Timeslot':<12}")
    print("-" * 55)
    for i in range(NUM_COURSES):
        room_idx = int(np.clip(round(best_position[i * 2]), 0, NUM_ROOMS - 1))
        slot_idx = int(np.clip(round(best_position[i * 2 + 1]), 0, NUM_TIMESLOTS - 1))
        print(f"{COURSES[i]:<10} | {PROFESSORS[i]:<12} | {ROOMS[room_idx]:<10} | {TIMESLOTS[slot_idx]:<12}")

if __name__ == "__main__":
    best_pos, best_fit = pso_timetable_optimizer(pop_size=40, max_iter=50)
    print_timetable(best_pos)
    print(f"\nFinal Schedule Penalty Score: {best_fit} (Lower is better)")


==== OPTIMIZED UNIVERSITY TIMETABLE ====
Course     | Professor    | Room       | Timeslot    
-------------------------------------------------------
CS101      | Prof. A      | Room 101   | Fri 9:00    
CS102      | Prof. B      | Room 102   | Wed 11:00   
MATH201    | Prof. A      | Room 102   | Wed 9:00    
PHY101     | Prof. C      | Room 101   | Wed 9:00    
BIO101     | Prof. D      | Room 101   | Wed 11:00   
ENG101     | Prof. B      | Room 101   | Mon 11:00   

Final Schedule Penalty Score: 0 (Lower is better)
